# AkidaNet vs Quantized Keras CNN on ImageNet-C

Benchmark AkidaNet on the AKD1000 PCIe card (or simulator fallback) against the quantized Keras baseline across ImageNet-C corruptions.

In [ ]:
import os, re, csv, time
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # MetaTF needs tf-keras / Keras 2

import numpy as np
import tensorflow as tf
import akida, cnn2snn, quantizeml, akida_models
from akida_models import akidanet_imagenet_pretrained
from akida_models.imagenet import get_preprocessed_samples
from cnn2snn import convert, set_akida_version, AkidaVersion

# Verify env
for m in (tf, akida, cnn2snn, quantizeml, akida_models):
    print(m.__name__, getattr(m, "__version__", "?"))

In [ ]:
# Model settings
ALPHA = 0.5  # width multiplier (0.5 @ 224x224)
IMG_SIZE = 224

# Find ImageNet-C
for candidate in ["../data/imagenet_subset/imagenet-c", "imagenet_subset/imagenet-c"]:
    if os.path.isdir(candidate):
        IMAGENET_C_DIR = candidate
        break
else:
    IMAGENET_C_DIR = "../data/imagenet_subset/imagenet-c"

IMAGENET_VAL_DIR = None 

CORRUPTIONS = ["fog", "gaussian_noise", "contrast"]
SEVERITIES = [1, 2, 3, 4, 5]
N_PER_CLASS = 2      # 2 images * 1000 classes = 2000 images per condition
CHUNK = 250          # Batch size for inference
MIN_AGREEMENT = 0.7  # Sanity threshold on initial batch
MAX_TOP1_GAP = 0.03  # Flag if card/sim differs from Keras by > 3%

# Use AKD1000 if present, else fallback to sim
FORCE_SIMULATOR = False
devices = akida.devices()
USE_HARDWARE = bool(devices) and not FORCE_SIMULATOR

RESULTS_DIR = "results_card" if USE_HARDWARE else "results_sim"
print(f"Backend: {'AKD1000 PCIe Card' if USE_HARDWARE else 'Akida CPU Simulator'}")
print(f"Results directory: {RESULTS_DIR}")

## 1. Pretrained quantized model

In [ ]:
# Load pretrained 4-bit AkidaNet (v1 architecture for AKD1000)
with set_akida_version(AkidaVersion.v1):
    model_keras = akidanet_imagenet_pretrained(ALPHA, quantized=True, bitwidth=4)
print("input :", model_keras.input.dtype, model_keras.input_shape)
print("output:", model_keras.output_shape)

# Sanity check on BrainChip sample images
x_demo, y_demo = get_preprocessed_samples(IMG_SIZE, 3)
pots = model_keras.predict(x_demo, batch_size=100)
acc_keras_demo = np.mean(np.argmax(pots, 1) == y_demo)
print(f"TF-Keras accuracy on the {len(x_demo)} sample images: {acc_keras_demo:.2f}")

## 2. Convert to Akida 1.0

In [ ]:
# Convert quantized Keras CNN to Akida SNN model
with set_akida_version(AkidaVersion.v1):
    model_akida = convert(model_keras)
print("akida IP version:", model_akida.ip_version)
model_akida.summary()
model_akida.save("akidanet_imagenet_a05_224_v1.fbz")
print("saved akidanet_imagenet_a05_224_v1.fbz")

# Dump layer config to CSV for inspection
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, "layers.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["index", "name", "layer_type", "input_dims", "output_dims", "input_bits", "weights_bits", "act_bits", "activation"])
    for i, l in enumerate(model_akida.layers):
        p = l.parameters
        get = lambda k: getattr(p, k) if k in dir(p) else ""
        w.writerow([i, l.name, str(p.layer_type).split(".")[-1], "x".join(map(str, l.input_dims)),
                    "x".join(map(str, l.output_dims)), l.input_bits,
                    get("weights_bits"), get("act_bits"), get("activation")])
print("wrote", os.path.join(RESULTS_DIR, "layers.csv"))

## 3. Akida verification & hardware check

In [ ]:
if USE_HARDWARE:
    device = devices[0]
    print("Available devices:", [d.desc for d in devices])
    assert device.version == akida.NSoC_v2, "Requires AKD1000 (NSoC_v2)"
    model_akida.map(device)
    model_akida.summary()
    device.soc.power_measurement_enabled = True
    print("Mapped to AKD1000 hardware card.")
else:
    print("Akida devices visible:", akida.devices(), "(running on software simulator)")

# Verify Akida model before running benchmark
acc_akida_demo = model_akida.evaluate(x_demo, y_demo)
pred_akida_demo = np.asarray(model_akida.predict(x_demo)).reshape(len(x_demo), -1).argmax(1)
backend_name = "AKD1000 card" if USE_HARDWARE else "Akida simulator"
print(f"{backend_name} accuracy on the {len(x_demo)} sample images: evaluate {acc_akida_demo:.2f}, "
      f"predict-argmax {np.mean(pred_akida_demo == y_demo):.2f}")
if USE_HARDWARE:
    print("Hardware statistics:", model_akida.statistics)

## 4. ImageNet-C evaluation

In [ ]:
HAVE_C = bool(IMAGENET_C_DIR) and os.path.isdir(IMAGENET_C_DIR)
HAVE_VAL = bool(IMAGENET_VAL_DIR) and os.path.isdir(IMAGENET_VAL_DIR)

# Sample n images per class from folder structure
def build_sample(ref_dir, n_per_class):
    wnids = sorted(d for d in os.listdir(ref_dir) if os.path.isdir(os.path.join(ref_dir, d)))
    assert len(wnids) == 1000, f"Expected 1000 class folders in {ref_dir}, found {len(wnids)}"
    sample = []
    for idx, wnid in enumerate(wnids):
        files = sorted(os.listdir(os.path.join(ref_dir, wnid)))[:n_per_class]
        sample += [(idx, wnid, f) for f in files]
    return sample

def load_image(path, clean):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    if clean:  # standard ImageNet resize (shorter side to 256) + center crop
        shape = tf.cast(tf.shape(img)[:2], tf.float32)
        new_hw = tf.cast(tf.round(shape * (256.0 / tf.reduce_min(shape))), tf.int32)
        img = tf.image.resize(img, new_hw, method=tf.image.ResizeMethod.BILINEAR)
        offset_y = (new_hw[0] - IMG_SIZE) // 2
        offset_x = (new_hw[1] - IMG_SIZE) // 2
        img = tf.image.crop_to_bounding_box(img, offset_y, offset_x, IMG_SIZE, IMG_SIZE)
    else:  # ImageNet-C images are already 224x224
        img = tf.ensure_shape(img, (IMG_SIZE, IMG_SIZE, 3))
    return tf.cast(img, tf.uint8)

def make_ds(paths, labels, clean):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, y: (load_image(p, clean), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(CHUNK).prefetch(tf.data.AUTOTUNE)

# Read energy per frame from hardware power meter
def read_energy():
    if not USE_HARDWARE:
        return None
    m = re.search(r"energy consumed \(mJ/frame\):\s*([0-9.]+)", str(model_akida.statistics))
    return float(m.group(1)) if m else None

first_chunk_checked = False

def evaluate_condition(paths, labels, clean):
    global first_chunk_checked
    keras_pred, akida_pred, energies = [], [], []
    for x, _ in make_ds(paths, labels, clean):
        x = x.numpy()
        keras_pred.append(np.asarray(model_keras.predict_on_batch(x)).reshape(len(x), -1).argmax(1))
        akida_pred.append(np.asarray(model_akida.predict(x)).reshape(len(x), -1).argmax(1))
        e = read_energy()
        if e is not None:
            energies.append(e)
        # Sanity check on the first batch
        if not first_chunk_checked:
            agree = float(np.mean(akida_pred[0] == keras_pred[0]))
            label = "card" if USE_HARDWARE else "sim"
            print(f"first chunk ({len(x)} images): {label} vs Keras agreement {agree:.3f}" + (f", energy {e} mJ/frame" if e is not None else ""))
            assert agree >= MIN_AGREEMENT, f"{label} vs Keras agreement {agree:.3f} < {MIN_AGREEMENT}"
            first_chunk_checked = True
    return np.concatenate(keras_pred), np.concatenate(akida_pred), (float(np.mean(energies)) if energies else None)


In [ ]:
rows_acc, rows_agree, rows_energy, rows_pred = [], [], [], []
ENERGY_SOURCE = f"AKD1000 power meter, model.statistics after model.predict on chunks of {CHUNK}, mJ/frame incl. floor power"
AKIDA_MODEL_NAME = "akidanet_akida_akd1000" if USE_HARDWARE else "akidanet_akida_simulator"

def write_csv(name, header, rows):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(os.path.join(RESULTS_DIR, name), "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)

def flush():
    write_csv("accuracy.csv", ["model", "corruption", "severity", "n_images", "top1"], rows_acc)
    write_csv("agreement.csv", ["corruption", "severity", "n_images", "agreement"], rows_agree)
    if USE_HARDWARE:
        write_csv("energy.csv", ["model", "corruption", "severity", "energy_per_inference", "source"], rows_energy)
    write_csv("predictions.csv", ["image_id", "corruption", "severity", "label", "keras_pred", "akida_pred"], rows_pred)

if not HAVE_C:
    print(f"IMAGENET_C_DIR ({IMAGENET_C_DIR}) not found, skipping evaluation.")
else:
    # Sample eval set from the first corruption cond as ref
    sample = build_sample(os.path.join(IMAGENET_C_DIR, CORRUPTIONS[0], str(SEVERITIES[0])), N_PER_CLASS)
    labels = np.array([s[0] for s in sample], dtype=np.int64)
    n = len(sample)
    print(f"{n} images per condition")

    # Eval both models on all corruptions and severities
    conditions = ([("clean", 0)] if HAVE_VAL else []) + [(c, s) for c in CORRUPTIONS for s in SEVERITIES]
    for corruption, severity in conditions:
        clean = corruption == "clean"
        root = IMAGENET_VAL_DIR if clean else os.path.join(IMAGENET_C_DIR, corruption, str(severity))
        paths = [os.path.join(root, w, f) for _, w, f in sample]
        assert os.path.isfile(paths[0]), f"missing file: {paths[0]}"
        t0 = time.time()
        k_pred, a_pred, energy = evaluate_condition(paths, labels, clean)

        k_top1 = float(np.mean(k_pred == labels))
        a_top1 = float(np.mean(a_pred == labels))
        agree  = float(np.mean(a_pred == k_pred))
        rows_acc.append(["akidanet_keras_quantized", corruption, severity, n, f"{k_top1:.4f}"])
        rows_acc.append([AKIDA_MODEL_NAME, corruption, severity, n, f"{a_top1:.4f}"])
        rows_agree.append([corruption, severity, n, f"{agree:.4f}"])
        if USE_HARDWARE:
            rows_energy.append([AKIDA_MODEL_NAME, corruption, severity,
                                "" if energy is None else f"{energy:.3f}",
                                ENERGY_SOURCE if energy is not None else "not reported by the SDK for this run"])
        for i, (_, w, f) in enumerate(sample):
            rows_pred.append([f"{w}/{f}", corruption, severity, int(labels[i]), int(k_pred[i]), int(a_pred[i])])
        
        energy_str = f"| energy {energy} mJ/frame " if energy is not None else ""
        backend_lbl = "card" if USE_HARDWARE else "sim"
        print(f"{corruption:15s} sev {severity}: keras {k_top1:.3f} | akida {backend_lbl} {a_top1:.3f} | agree {agree:.3f} "
              f"{energy_str} ({time.time() - t0:.0f}s)")
        if abs(a_top1 - k_top1) > MAX_TOP1_GAP:
            print(f"  WARNING: {backend_lbl} vs Keras top-1 gap {a_top1 - k_top1:+.3f} is larger than {MAX_TOP1_GAP}")
        flush()
        if clean or severity == 1:
            assert k_top1 > 0.2, f"Keras top-1 unexpectedly low: {k_top1:.3f}"
    written_files = ["accuracy.csv", "agreement.csv", "predictions.csv"] + (["energy.csv"] if USE_HARDWARE else [])
    print("wrote", [os.path.join(RESULTS_DIR, f) for f in written_files])